# Task 2: Transform Data

Reads previous task result via `taskValues`, applies transformations (duration, cost per mile) and passes the averages on as task values. Returns row count.

> Same code as in `notebooks/modules/05_orchestration_jobs.ipynb`. `dbutils.notebook.exit()` stays in the last cell on purpose — anything printed in that cell would be hidden in the job run output.


In [0]:
# TASK 2: Transform Data

from pyspark.sql.functions import *
from datetime import date
import json

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("run_date", "")

source_table = dbutils.widgets.get("source_table")
run_date = dbutils.widgets.get("run_date") or date.today().isoformat()

# Get the task value set by the upstream validate task.
# taskKey = the task name in the job; set the "validate_task" parameter if you named the task differently.
# (debugValue is returned when the notebook runs interactively, outside a job)
dbutils.widgets.text("validate_task", "validate")
validate_task = dbutils.widgets.get("validate_task")
try:
    validated_rows = dbutils.jobs.taskValues.get(taskKey=validate_task, key="row_count", debugValue=-1)
except ValueError:
    print(f"WARNING: no task '{validate_task}' in this job run - set parameter validate_task to the validate task name")
    validated_rows = -1
print(f"Rows validated by upstream task: {validated_rows}")

# Transformation
print(f"Transforming: {source_table}")

df = spark.table(source_table)

df_transformed = (
    df
    .withColumn("trip_duration_minutes", 
                round((col("tpep_dropoff_datetime").cast("long") - 
                       col("tpep_pickup_datetime").cast("long")) / 60, 2))
    .withColumn("cost_per_mile", 
                when(col("trip_distance") > 0, 
                     round(col("fare_amount") / col("trip_distance"), 2))
                .otherwise(0))
    .withColumn("processing_date", lit(run_date))
)

row_count = df_transformed.count()
print(f"Transformed {row_count} rows")

# Pass the averages to the report task as task values
stats = df_transformed.agg(
    round(avg("trip_duration_minutes"), 2).alias("avg_trip_minutes"),
    round(avg("cost_per_mile"), 2).alias("avg_cost_per_mile"),
).first()
dbutils.jobs.taskValues.set(key="avg_trip_minutes", value=float(stats.avg_trip_minutes))
dbutils.jobs.taskValues.set(key="avg_cost_per_mile", value=float(stats.avg_cost_per_mile))
print(f"Avg trip: {stats.avg_trip_minutes:.2f} min | Avg cost per mile: ${stats.avg_cost_per_mile:.2f}")

df_transformed.select(
    "trip_distance", "fare_amount", "trip_duration_minutes", "cost_per_mile"
).show(5)


In [0]:
# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "rows_transformed": row_count
}))